# series/generative-adversarial-networks/05 GAN From Scratch

Part 5 of the GAN series. Fresh Python 3.11+ CPU kernel; official full Fashion-MNIST training array, no API key. Executes all1000 source random-batch iterations (called epochs in source), not1000 complete dataset passes or the historical10000-iteration figure. Fix current imports, malformed code/prose boundary, missing closing parenthesis, dtype, separate optimizers and explicit discriminator freezing. All20 source-cadence100-image grids saved. User explicitly approved changing the final generator activation from ReLU to tanh on October26,2026, matching real-image normalization to[-1,1]. Every hidden ReLU and the remaining architecture stay unchanged. Original unmodified ReLU code remains in appendix. Source historical figures are not new run results.

In [ ]:
import os
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "tensorflow==2.20.0", "keras==3.15.1", "numpy==2.4.6", "certifi==2026.10.23"], check=True)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Sequential, Input
from tensorflow.keras.layers import Conv2D, LeakyReLU, Dropout, ZeroPadding2D, BatchNormalization, Flatten, Dense, Reshape, UpSampling2D
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
keras.utils.set_random_seed(42)
np.set_printoptions(precision=6, suppress=True)
print("Python:", sys.version.split()[0], "| TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
print("Matplotlib and filesystem helpers ready.")

## Example 0

In [ ]:
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.optimizers import Adam
print("Fashion-MNIST and current Keras layers/optimizers imported.")

## Example 1

In [ ]:
(training_data, _), (_, _) = fashion_mnist.load_data()
X_train = training_data.astype("float32") / 127.5 - 1.0
X_train = np.expand_dims(X_train, axis=3)
print("Full training set:", X_train.shape, X_train.dtype)
print("Normalized range:", float(X_train.min()), float(X_train.max()))

## Example 2

In [ ]:
def visualize_input(img, ax):
    ax.imshow(img, cmap='gray')
    width, height = img.shape
    thresh = img.max()/2.5
    for x in range(width):
        for y in range(height):
            ax.annotate(str(round(img[x][y],2)), xy=(y,x),
                        horizontalalignment='center',
                        verticalalignment='center',
                        color='white' if img[x][y]<thresh else 'black')        
        
fig = plt.figure(figsize = (12,12))
ax = fig.add_subplot(111)
visualize_input(training_data[3343], ax)
plt.show()
print("Annotated Fashion-MNIST image index:", 3343)

## Example 3

In [ ]:
def build_generator():
    generator = Sequential([
        Input(shape=(100,)),
        Dense(128 * 7 * 7, activation="relu"),
        Reshape((7, 7, 128)), UpSampling2D(size=(2, 2)),
        Conv2D(128, kernel_size=3, padding="same", activation="relu"),
        BatchNormalization(momentum=0.8),
        UpSampling2D(size=(2, 2)),
        Conv2D(64, kernel_size=3, padding="same", activation="relu"),
        BatchNormalization(momentum=0.8),
        Conv2D(1, kernel_size=3, padding="same", activation="tanh"),
    ], name="generator")
    return generator

generator = build_generator()
print("Generator input/output:", generator.input_shape, generator.output_shape)
print("Generator parameters:", generator.count_params())

## Example 4

In [ ]:
def build_discriminator():
    discriminator = Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, kernel_size=3, strides=2, padding="same"),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Conv2D(64, kernel_size=3, strides=2, padding="same"),
        ZeroPadding2D(padding=((0, 1), (0, 1))),
        BatchNormalization(momentum=0.8),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Conv2D(128, kernel_size=3, strides=2, padding="same"),
        BatchNormalization(momentum=0.8),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Conv2D(256, kernel_size=3, strides=1, padding="same"),
        BatchNormalization(momentum=0.8),
        LeakyReLU(negative_slope=0.2), Dropout(0.25),
        Flatten(), Dense(1, activation="sigmoid"),
    ], name="discriminator")
    return discriminator

discriminator = build_discriminator()
print("Discriminator input/output:", discriminator.input_shape, discriminator.output_shape)
print("Discriminator parameters:", discriminator.count_params())

## Example 5

In [ ]:
def configure_gan():
    discriminator = build_discriminator()
    discriminator.compile(loss="binary_crossentropy", optimizer=Adam(learning_rate=0.0002, beta_1=0.5), metrics=["accuracy"])
    generator = build_generator()
    discriminator.trainable = False
    z = Input(shape=(100,))
    image = generator(z)
    validity = discriminator(image, training=False)
    combined = Model(z, validity)
    combined.compile(loss="binary_crossentropy", optimizer=Adam(learning_rate=0.0002, beta_1=0.5))
    return discriminator, generator, combined
discriminator, generator, combined = configure_gan()
print("Separate optimizers:", discriminator.optimizer is not combined.optimizer)
print("Combined output:", combined.output_shape)

## Example 6

In [ ]:
def train(epochs, batch_size=128, save_interval=50):
    valid = np.ones((batch_size, 1), dtype="float32")
    fake = np.zeros((batch_size, 1), dtype="float32")
    history = []
    for epoch in range(epochs):
        idx = np.random.randint(0, X_train.shape[0], batch_size)
        imgs = X_train[idx]
        noise = np.random.normal(0, 1, (batch_size, latent_dim)).astype("float32")
        gen_imgs = generator(noise, training=False).numpy()
        check_weights = epoch % save_interval == 0 or epoch == epochs - 1
        if check_weights:
            old_g = [v.numpy().copy() for v in generator.weights]
            old_d = [v.numpy().copy() for v in discriminator.weights]
        discriminator.trainable = True
        discriminator.reset_metrics()
        d_loss_real = discriminator.train_on_batch(imgs, valid)
        discriminator.reset_metrics()
        d_loss_fake = discriminator.train_on_batch(gen_imgs, fake)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
        if check_weights:
            assert all(np.array_equal(v.numpy(), old) for v, old in zip(generator.weights, old_g))
            assert any(not np.array_equal(v.numpy(), old) for v, old in zip(discriminator.weights, old_d))
            frozen_d = [v.numpy().copy() for v in discriminator.weights]
            old_g_trainable = [v.numpy().copy() for v in generator.trainable_weights]
        discriminator.trainable = False
        combined.reset_metrics()
        g_loss = float(combined.train_on_batch(noise, valid))
        assert np.isfinite(d_loss).all() and np.isfinite(g_loss)
        if check_weights:
            assert all(np.array_equal(v.numpy(), old) for v, old in zip(discriminator.weights, frozen_d))
            assert any(not np.array_equal(v.numpy(), old) for v, old in zip(generator.trainable_weights, old_g_trainable))
        history.append([float(d_loss[0]), float(d_loss[1]), g_loss])
        if epoch % save_interval == 0 or epoch == epochs - 1:
            print(f"{epoch} [D loss: {d_loss[0]:.6f}, acc.: {100*d_loss[1]:.2f}%] [G loss: {g_loss:.6f}]")
        if epoch % save_interval == 0:
            plot_generated_images(epoch, generator)
    return np.asarray(history)
print("Alternating training function ready.")

## Example 7

In [ ]:
def plot_generated_images(epoch, generator, examples=100, dim=(10, 10), figsize=(10, 10)):
    noise = np.random.normal(0, 1, size=(examples, latent_dim)).astype("float32")
    generated_images = generator(noise, training=False).numpy().reshape(examples, 28, 28)
    fig, axes = plt.subplots(*dim, figsize=figsize)
    for ax, image in zip(axes.flat, generated_images):
        ax.imshow(image, interpolation="nearest", cmap="gray_r")
        ax.axis("off")
    fig.tight_layout()
    filename = f"gan_generated_image_epoch_{epoch}.png"
    fig.savefig(filename)
    plt.close(fig)
    return filename
print("100-image sample-grid writer ready.")

## Example 8

In [ ]:
img_shape = (28, 28, 1)
channels = 1
latent_dim = 100
discriminator, generator, combined = configure_gan()
assert {id(v) for v in combined.trainable_weights} == {id(v) for v in generator.trainable_weights}
print("Image shape:", img_shape, "| latent dimension:", latent_dim)
print("Trainable combined parameters:", sum(int(np.prod(v.shape)) for v in combined.trainable_weights))

## Example 9

In [ ]:
history = train(epochs=1000, batch_size=32, save_interval=50)
print("Completed training iterations:", len(history))
print("Discriminator updates:", int(discriminator.optimizer.iterations.numpy()))
print("Generator updates:", int(combined.optimizer.iterations.numpy()))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, step in zip(axes, [0, 50, 950]):
    ax.imshow(Image.open(f"gan_generated_image_epoch_{step}.png"))
    ax.set_title(f"Iteration {step}")
    ax.axis("off")
fig.tight_layout()
plt.show()

## Independent regression checks

In [ ]:
assert X_train.shape == (60000,28,28,1) and X_train.dtype == np.float32
assert history.shape == (1000,3) and np.isfinite(history).all()
assert int(discriminator.optimizer.iterations.numpy()) == 2000
assert int(combined.optimizer.iterations.numpy()) == 1000
files = [Path(f"gan_generated_image_epoch_{i}.png") for i in range(0,1000,50)]
assert len(files) == 20 and all(p.is_file() for p in files)
for p in files:
    with Image.open(p) as image:
        assert image.size == (1000,1000)
probe = np.random.default_rng(7).normal(size=(4,100)).astype("float32")
generated = generator(probe, training=False).numpy()
assert generated.shape == (4,28,28,1) and np.isfinite(generated).all()
assert (generated >= -1).all() and (generated <= 1).all()
assert generator.layers[-1].activation.__name__ == "tanh"
print("All 1000 source training iterations, 2000 discriminator/1000 generator updates, 20 complete100-image grids and repeated frozen-weight checks passed.")
print("Final D loss / accuracy / G loss:", history[-1])
print("Generated pixel range:", float(generated.min()), float(generated.max()))

## Original source appendix

Full original blocks from raw source SHA-256 0e3c81adcb17fbd6782fd408316d05820ac231838e23ba885db45528c81d58c2. Preserved verbatim as nonexecuting historical material, including mistakes/legacy setup alternatives.

### Original block 1

```text
from __future__ import print_function, division
from keras.datasets import fashion_mnist
from keras.layers import Input, Dense, Reshape, Flatten, Dropout
from keras.layers import BatchNormalization, Activation, ZeroPadding2D
from keras.layers.advanced_activations import LeakyReLU
from keras.layers.convolutional import UpSampling2D, Conv2D
from keras.models import Sequential, Model
from keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
```

### Original block 2

```text
(training_data, _), (_, _) = fashion_mnist.load_data()
X_train = training_data / 127.5 - 1.
X_train = np.expand_dims(X_train, axis=3)
```

### Original block 3

```text
def visualize_input(img, ax):
    ax.imshow(img, cmap='gray')
    width, height = img.shape
    thresh = img.max()/2.5
    for x in range(width):
        for y in range(height):
            ax.annotate(str(round(img[x][y],2)), xy=(y,x),
                        horizontalalignment='center',
                        verticalalignment='center',
                        color='white' if img[x][y]<thresh else 'black')        
        
fig = plt.figure(figsize = (12,12))
ax = fig.add_subplot(111)
visualize_input(training_data[3343], ax)We also use batch normalization and a ReLU activation.
For each of these layers, the general scheme is convolution ⇒ batch normalization
⇒ ReLU. We keep stacking up layers like this until we get the final transposed
convolution layer with shape 28 × 28 × 1:
```

### Original block 4

```text
def build_generator():
  
  generator = Sequential()
  
  generator.add(Dense(6272, activation="relu", input_dim=100)) # Add dense layer
  generator.add(Reshape((7, 7, 128)))  # reshape the image
  generator.add(UpSampling2D()) # Upsampling layer to double the size of the image
  generator.add(Conv2D(128, kernel_size=3, padding="same", activation="relu"))
  generator.add(BatchNormalization(momentum=0.8))
  generator.add(UpSampling2D())
  
  # convolutional + batch normalization layers
  generator.add(Conv2D(64, kernel_size=3, padding="same", activation="relu"))
  generator.add(BatchNormalization(momentum=0.8))
  
  # convolutional layer with filters = 1
  generator.add(Conv2D(1, kernel_size=3, padding="same", activation="relu"))
  generator.summary() # prints the model summary
  
  """
  We don't add upsampling here because the image size of 28 × 28 is 
  equal to the image size in the MNIST dataset. 
  You can adjust this for your own problem.
  """
  noise = Input(shape=(100,))
  fake_image = generator(noise)
  
  # Returns a model that takes the noise vector as an input and outputs the fake image
  return Model(inputs=noise, outputs=fake_image)
```

### Original block 5

```text
def build_discriminator():
  discriminator = Sequential()
  discriminator.add(Conv2D(32, kernel_size=3, strides=2, input_shape=(28,28,1), padding="same"))
  discriminator.add(LeakyReLU(alpha=0.2))
  discriminator.add(Dropout(0.25))
  
  discriminator.add(Conv2D(64, kernel_size=3, strides=2,padding="same"))
  discriminator.add(ZeroPadding2D(padding=((0,1),(0,1))))
  discriminator.add(BatchNormalization(momentum=0.8))
  
  discriminator.add(LeakyReLU(alpha=0.2))
  discriminator.add(Dropout(0.25))
  
  discriminator.add(Conv2D(128, kernel_size=3, strides=2, padding="same"))
  discriminator.add(BatchNormalization(momentum=0.8))
  discriminator.add(LeakyReLU(alpha=0.2))
  discriminator.add(Dropout(0.25))
  
  discriminator.add(Conv2D(256, kernel_size=3, strides=1, padding="same"))
  discriminator.add(BatchNormalization(momentum=0.8))
  discriminator.add(LeakyReLU(alpha=0.2))
  discriminator.add(Dropout(0.25))
  
  discriminator.add(Flatten())
  discriminator.add(Dense(1, activation='sigmoid'))
  
  img = Input(shape=(28,28,1))
  probability = discriminator(img)
 
  return Model(inputs=img, outputs=probability)
```

### Original block 6

```text
optimizer = Adam(learning_rate=0.0002, beta_1=0.5)
discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
discriminator.trainable = False

# Build the generator
generator = build_generator()
z = Input(shape=(100,))
img = generator(z)
valid = discriminator(img)
combined = Model(inputs=z, outputs=valid)
combined.compile(loss='binary_crossentropy', optimizer=optimizer)
```

### Original block 7

```text
def train(epochs, batch_size=128, save_interval=50):
    
    valid = np.ones((batch_size, 1))
    fake = np.zeros((batch_size, 1))
    
    for epoch in range(epochs):  # Train Discriminator network
        idx = np.random.randint(0, X_train.shape[0], batch_size)
        imgs = X_train[idx]
        
        noise = np.random.normal(0, 1, (batch_size, 100))
        gen_imgs = generator.predict(noise)
        
        d_loss_real = discriminator.train_on_batch(imgs, valid)
        d_loss_fake = discriminator.train_on_batch(gen_imgs, fake)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
        
        g_loss = combined.train_on_batch(noise, valid)
        
        # printing progress
        print("%d [D loss: %f, acc.: %.2f%%] [G loss: %f]" %(epoch, d_loss[0], 100*d_loss[1], g_loss))
        
        if epoch % save_interval == 0:
            plot_generated_images(epoch, generator)
```

### Original block 8

```text
def plot_generated_images(epoch, generator, examples=100, dim=(10, 10),figsize=(10, 10)):
    noise = np.random.normal(0, 1, size=[examples, latent_dim])
    generated_images = generator.predict(noise)
    generated_images = generated_images.reshape(examples, 28, 28)
    
    plt.figure(figsize=figsize)
    for i in range(generated_images.shape[0]):
        plt.subplot(dim[0], dim[1], i+1)
        plt.imshow(generated_images[i], interpolation='nearest', cmap='gray_r')
        plt.axis('off')
    
    plt.tight_layout()
    plt.savefig('gan_generated_image_epoch_%d.png' % epoch
```

### Original block 9

```text
# Input shape
img_shape = (28,28,1)
channels = 1
latent_dim = 100
optimizer = Adam(0.0002, 0.5)

# Build and compile the discriminator
discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
# Build the generator
generator = build_generator()
# The generator takes noise as input and generates imgs
z = Input(shape=(latent_dim,))
img = generator(z)
# For the combined model we will only train the generator
discriminator.trainable = False
# The discriminator takes generated images as input and determines validity
valid = discriminator(img)
# The combined model  (stacked generator and discriminator)
# Trains the generator to fool the discriminator
combined = Model(z, valid)
combined.compile(loss='binary_crossentropy', optimizer=optimizer)
```

### Original block 10

```text
train(epochs=1000, batch_size=32, save_interval=50)
```